<a href="https://colab.research.google.com/github/irullah/comic-yolo/blob/main/Training_AI_Comic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
import os
import glob
import yaml
from roboflow import Roboflow

# ==============================================================================
# TAHAP 1: UNDUH DATASET V3 DARI ROBOFLOW
# ==============================================================================
print("[INFO] Mengunduh dataset v3 dari Roboflow...")
rf = Roboflow(api_key="w7A0SzgQrYyWSGqS4CaX")
project = rf.workspace("poos-workspace-20cho").project("ruru-u8tn6")

# Roboflow menggunakan nomor urut bulat: v1 = 1, v2 = 2, v3 = 3
version = project.version(3)
dataset = version.download("yolov8")

# ==============================================================================
# TAHAP 2: VALIDASI DATA.YAML (TANPA POTONG ULANG)
# ==============================================================================
src_dir = dataset.location
yaml_path = os.path.join(src_dir, "data.yaml")

if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        data_yaml = yaml.safe_load(f)

    # Pastikan lokasi path absolut agar YOLO tidak tersesat
    data_yaml['path'] = os.path.abspath(src_dir)
    data_yaml['train'] = 'train/images'

    # Cek ketersediaan gambar di folder validasi
    val_images = glob.glob(os.path.join(src_dir, 'valid', 'images', '*.*'))
    if len(val_images) > 0:
        data_yaml['val'] = 'valid/images'
        print(f"✅ [INFO] Data validasi ditemukan ({len(val_images)} gambar)!")
    else:
        print("🔧 [PERBAIKAN] Folder 'valid' kosong. Mengalihkan validasi ke 'train'...")
        data_yaml['val'] = 'train/images'

    if 'test' in data_yaml:
        data_yaml['test'] = 'test/images'

    with open(yaml_path, 'w') as f:
        yaml.dump(data_yaml, f)

print(f"\n✅ [INFO] Dataset v3 siap digunakan di: {src_dir}")
print("[INFO] Silakan lanjut jalankan Block 3!")

In [ ]:
import os
import zipfile
from google.colab import files

# Nama file hasil kompresi
zip_filename = 'hasil_training_ruru_v3.zip'

# Mengompres folder hasil training (berisi best.pt, last.pt, dan grafik akurasi)
# Folder dataset gambar tidak ikut di-zip agar ukuran file ringan & cepat diunduh
items_to_zip = ['Comic_Cleaner', 'runs']

print("📦 Sedang mengompres folder hasil training...")

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for item in items_to_zip:
        if os.path.isdir(item):
            for root, dirs, filenames in os.walk(item):
                for filename in filenames:
                    file_path = os.path.join(root, filename)
                    zipf.write(file_path, file_path)
        elif os.path.exists(item):
            zipf.write(item, item)

print(f"✅ Proses kompresi selesai! File {zip_filename} sedang diunduh...")
files.download(zip_filename)